# 03 — ÉVALUATION FINALE sur test.csv

Responsable : **Jeanne**

> ⚠️ Ce notebook est exécuté **une seule fois**, après le gel du choix du modèle (synchro 3).
> `test.csv` représente les passagers du mois prochain : il n'a servi ni à explorer, ni à choisir les variables, ni à régler le modèle.

Date et heure de l'exécution unique : voir la sortie de la cellule 2

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))  # pour importer src/ depuis notebooks/

from src.data import load_train, load_test, TARGET, RANDOM_STATE
from src.features import LANDING_FEATURES
from src.economics import gain_net

## 1. Ré-entraînement du modèle retenu sur tout train

In [2]:
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

from src.features import NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, MODEL_FEATURES, AGE_MINIMUM_SIGNALEMENT


def preparer(df):
    """Mêmes traitements que dans le notebook 02."""
    X = df[MODEL_FEATURES].copy()
    X["Arrival Delay in Minutes"] = X["Arrival Delay in Minutes"].fillna(X["Departure Delay in Minutes"])
    return X


# Le modèle retenu, réentraîné sur la totalité de train
train = load_train()
X_train, y_train = preparer(train), train[TARGET]

modele = make_pipeline(
    ColumnTransformer([
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), MODEL_CATEGORICAL_FEATURES),
    ]),
    HistGradientBoostingClassifier(random_state=RANDOM_STATE),
)
modele.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('columntransformer', ...), ('histgradientboostingclassifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](6,)","['Age','Flight Distance','Departure Delay in Minutes', 'Arrival Delay in Minutes','Customer Type','Class']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,6
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default

## 2. Application unique à test et performance finale

In [3]:
# ===== ÉVALUATION FINALE — test.csv lu ici pour la première et unique fois =====
print("Exécution :", datetime.now().strftime("%Y-%m-%d %H:%M"))

test = load_test()
X_test, y_test = preparer(test), test[TARGET]

scores_test = modele.predict_proba(X_test)[:, 1]
mineur_test = (test["Age"] < AGE_MINIMUM_SIGNALEMENT).values   # jamais signalés

# Mêmes règles de comparaison que dans le notebook 02, mineurs exclus partout
rng = np.random.default_rng(RANDOM_STATE)
eco = (X_test["Class"] == "Eco").astype(int)

final = pd.DataFrame([
    gain_net(y_test, np.where(mineur_test, -np.inf, rng.random(len(y_test)))),
    gain_net(y_test, np.where(mineur_test, -np.inf, eco * 10_000 + X_test["Arrival Delay in Minutes"])),
    gain_net(y_test, np.where(mineur_test, -np.inf, scores_test)),
], index=["Hasard", "Règle : Eco puis retard", "Modèle retenu"])

final[["passagers_abordes", "insatisfaits_abordes", "precision", "gain_net_pour_1000"]].round(3)

Exécution : 2026-10-14 15:05


,passagers_abordes,insatisfaits_abordes,precision,gain_net_pour_1000
Hasard,2597,1410,0.543,1429.011
Règle : Eco puis retard,2597,2197,0.846,4458.731
Modèle retenu,2597,2435,0.938,5374.962


## 3. Lecture des résultats

### Ce qu'on a mesuré
Le modèle retenu (gradient boosting, sans le type de voyage, mineurs jamais signalés) a été réentraîné sur la totalité de train, puis appliqué **une seule fois** aux 25 976 passagers de test, qui représentent les passagers du mois prochain. Aucun choix (variables, modèle, réglages) n'a été fait en regardant ce fichier : le code a été committé avant son exécution.

### Résultat final

| Sur test.csv | Précision | Gain net pour 1 000 passagers |
|---|---|---|
| Hasard | 0,543 | 1 429 € |
| Règle simple : Eco puis retard | 0,846 | 4 459 € |
| **Modèle retenu** | **0,938** | **5 375 €** |

### Ce que ça veut dire concrètement
Pour 1 000 passagers qui atterrissent, les équipes en abordent 100, choisis par le modèle :
- **94 sont réellement neutres ou insatisfaits** ; seules 6 approches sont inutiles ;
- aborder ces 100 passagers coûte 4 000 €, et évite de perdre environ 9 400 € de revenu futur ;
- **gain net : environ 5 400 € pour 1 000 passagers arrivants**, soit 3,8 fois plus qu'en abordant des passagers au hasard, et 916 € de plus que la meilleure règle applicable sans modèle ;
- **aucun mineur n'est signalé**.

### Le résultat est fiable
| | Précision | Gain pour 1 000 passagers |
|---|---|---|
| Validation croisée sur train (notebook 02) | 0,939 | 5 386 € |
| Évaluation finale sur test | 0,938 | 5 375 € |

Les deux mesures sont quasi identiques : sur des passagers jamais vus, le modèle se comporte comme prévu. La performance annoncée à la direction n'est pas surestimée.

### Limites
- **Hypothèses de la direction** : le gain dépend des estimations fournies (40 € par approche, 1 mécontent sur 3 perdu, 600 € de revenu futur, la moitié récupérée par une approche). Si elles changent, le gain change, mais le classement des options reste le même tant que la précision dépasse 40 %.
- **Répondants vs tous les passagers** : le modèle est entraîné sur des passagers qui ont répondu à l'enquête, et il serait appliqué à tous les passagers arrivants. Si les non-répondants sont différents, la performance réelle peut différer : il faudra la suivre après le déploiement.
- **Capacité** : avec 10 % de passagers abordables, environ 470 mécontents sur 1 000 ne seront pas abordés, quel que soit le modèle. Pour eux, le levier est l'amélioration du service (notebook 01).
- **Hypothèse sur les variables** : on suppose que le type de client (fidèle / non fidèle) correspond au statut du programme de fidélité, connu de la compagnie.

### Recommandation
Déployer le modèle **à titre pilote** sur quelques aéroports, aux conditions de protection des données décrites dans le rapport (pas de mineurs signalés, information minimale montrée aux agents), et **comparer** pendant un mois les passagers abordés à un groupe témoin non abordé, pour vérifier le taux de retour réel.